In [ ]:
!pip -q install gradio

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import joblib
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import gradio as gr

from pathlib import Path
from PIL import Image
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# =========================
# Paths
# =========================

BASE_DIR = Path("/content/drive/MyDrive/speedcube-colour-drifting")
MODELS_DIR = BASE_DIR / "models"

EFFNET_MLP_MODEL_PATH = MODELS_DIR / "efficientnetv2_mlp_head.pt"
EFFNET_MLP_SCALER_PATH = MODELS_DIR / "efficientnetv2_mlp_scaler.pkl"

print("Model exists :", EFFNET_MLP_MODEL_PATH.exists())
print("Scaler exists:", EFFNET_MLP_SCALER_PATH.exists())


# =========================
# Device
# =========================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# =========================
# MLP Head Definition
# =========================

class EfficientNetMLPHead(nn.Module):
    def __init__(self, input_dim=1280, num_classes=6, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# =========================
# Load checkpoint
# =========================

try:
    checkpoint = torch.load(
        EFFNET_MLP_MODEL_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        EFFNET_MLP_MODEL_PATH,
        map_location=DEVICE,
    )

effnet_scaler = joblib.load(EFFNET_MLP_SCALER_PATH)

classes = list(checkpoint["classes"])
input_dim = checkpoint["input_dim"]
num_classes = checkpoint["num_classes"]
dropout = checkpoint["dropout"]

print("Classes:", classes)


# =========================
# Load EfficientNetV2-S feature extractor
# =========================

efficientnet_weights = EfficientNet_V2_S_Weights.DEFAULT
efficientnet_preprocess = efficientnet_weights.transforms()

efficientnet = efficientnet_v2_s(weights=efficientnet_weights)

efficientnet_feature_extractor = nn.Sequential(
    efficientnet.features,
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
).to(DEVICE)

for param in efficientnet_feature_extractor.parameters():
    param.requires_grad = False

efficientnet_feature_extractor.eval()


# =========================
# Load MLP head
# =========================

effnet_mlp_model = EfficientNetMLPHead(
    input_dim=input_dim,
    num_classes=num_classes,
    dropout=dropout,
).to(DEVICE)

effnet_mlp_model.load_state_dict(checkpoint["model_state_dict"])
effnet_mlp_model.eval()

print("EfficientNetV2-S + MLP loaded successfully.")

Model exists : True
Scaler exists: True
Device: cpu
Classes: ['blue', 'green', 'orange', 'red', 'white', 'yellow']
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 89.0MB/s]


EfficientNetV2-S + MLP loaded successfully.


In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image


def order_points(pts):
    """
    Urutkan 4 titik menjadi:
    top-left, top-right, bottom-right, bottom-left
    """
    pts = np.array(pts, dtype=np.float32)

    rect = np.zeros((4, 2), dtype=np.float32)

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    rect[0] = pts[np.argmin(s)]      # top-left
    rect[2] = pts[np.argmax(s)]      # bottom-right
    rect[1] = pts[np.argmin(diff)]   # top-right
    rect[3] = pts[np.argmax(diff)]   # bottom-left

    return rect


def resize_for_detection(image_rgb, max_side=900):
    h, w = image_rgb.shape[:2]

    scale = 1.0
    if max(h, w) > max_side:
        scale = max_side / max(h, w)
        new_w = int(w * scale)
        new_h = int(h * scale)
        resized = cv2.resize(image_rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)
        return resized, scale

    return image_rgb.copy(), scale


def detect_rubik_face_quad(image_rgb):
    """
    Deteksi bidang Rubik sebagai 4 titik sudut.
    Metode:
    - edge detection
    - contour detection
    - approx polygon / minAreaRect fallback
    """
    small, scale = resize_for_detection(image_rgb, max_side=900)

    gray = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(gray, 50, 150)

    kernel = np.ones((5, 5), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    img_area = small.shape[0] * small.shape[1]
    candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < 0.03 * img_area:
            continue

        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.03 * peri, True)

        if len(approx) == 4 and cv2.isContourConvex(approx):
            pts = approx.reshape(4, 2).astype(np.float32)
        else:
            rect = cv2.minAreaRect(cnt)
            pts = cv2.boxPoints(rect).astype(np.float32)

        ordered = order_points(pts)

        width_top = np.linalg.norm(ordered[1] - ordered[0])
        width_bottom = np.linalg.norm(ordered[2] - ordered[3])
        height_left = np.linalg.norm(ordered[3] - ordered[0])
        height_right = np.linalg.norm(ordered[2] - ordered[1])

        avg_width = (width_top + width_bottom) / 2
        avg_height = (height_left + height_right) / 2

        if min(avg_width, avg_height) <= 0:
            continue

        aspect_ratio = max(avg_width, avg_height) / min(avg_width, avg_height)

        # Sisi Rubik harusnya mendekati kotak, tapi toleransi dibuat longgar
        if aspect_ratio > 1.8:
            continue

        score = area / aspect_ratio
        candidates.append((score, ordered))

    if len(candidates) == 0:
        raise ValueError("Bidang Rubik tidak terdeteksi. Coba gunakan background lebih kontras atau foto lebih dekat.")

    candidates.sort(key=lambda x: x[0], reverse=True)

    best_pts_small = candidates[0][1]
    best_pts_original = best_pts_small / scale

    return best_pts_original


def warp_rubik_face(image_rgb, quad_points, output_size=600):
    """
    Perspective transform dari bidang miring menjadi kotak lurus.
    """
    src = order_points(quad_points)

    dst = np.array([
        [0, 0],
        [output_size - 1, 0],
        [output_size - 1, output_size - 1],
        [0, output_size - 1],
    ], dtype=np.float32)

    matrix = cv2.getPerspectiveTransform(src, dst)

    warped = cv2.warpPerspective(
        image_rgb,
        matrix,
        (output_size, output_size),
    )

    return warped


def center_square_fallback(image_rgb, output_size=600):
    """
    Fallback kalau deteksi contour gagal:
    crop kotak dari tengah gambar.
    """
    h, w = image_rgb.shape[:2]
    side = min(h, w)

    x1 = (w - side) // 2
    y1 = (h - side) // 2
    x2 = x1 + side
    y2 = y1 + side

    crop = image_rgb[y1:y2, x1:x2]

    crop = cv2.resize(
        crop,
        (output_size, output_size),
        interpolation=cv2.INTER_AREA,
    )

    return crop


def split_warped_face_to_rois(warped_rgb, grid_size=3, crop_ratio=0.72):
    """
    Split wajah Rubik 3x3 menjadi 9 ROI.
    Center crop dipakai agar garis pembatas tidak ikut dominan.
    """
    h, w = warped_rgb.shape[:2]

    cell_h = h // grid_size
    cell_w = w // grid_size

    rois = []

    for row in range(grid_size):
        for col in range(grid_size):
            y1 = row * cell_h
            y2 = (row + 1) * cell_h
            x1 = col * cell_w
            x2 = (col + 1) * cell_w

            cell = warped_rgb[y1:y2, x1:x2]

            ch, cw = cell.shape[:2]

            crop_h = int(ch * crop_ratio)
            crop_w = int(cw * crop_ratio)

            cy1 = (ch - crop_h) // 2
            cx1 = (cw - crop_w) // 2

            roi = cell[
                cy1:cy1 + crop_h,
                cx1:cx1 + crop_w,
            ]

            rois.append({
                "row": row,
                "col": col,
                "image": Image.fromarray(roi),
            })

    return rois


def draw_prediction_grid(warped_rgb, predictions, output_size=600):
    """
    Buat preview gambar 3x3 dengan label prediksi.
    """
    preview = warped_rgb.copy()

    h, w = preview.shape[:2]
    cell_h = h // 3
    cell_w = w // 3

    for item in predictions:
        row = item["row"]
        col = item["col"]
        label = item["label"]
        confidence = item["confidence"]

        x1 = col * cell_w
        y1 = row * cell_h
        x2 = (col + 1) * cell_w
        y2 = (row + 1) * cell_h

        cv2.rectangle(
            preview,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2,
        )

        text = f"{label} {confidence:.2f}"

        cv2.putText(
            preview,
            text,
            (x1 + 8, y1 + 28),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        cv2.putText(
            preview,
            text,
            (x1 + 8, y1 + 28),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 0),
            1,
            cv2.LINE_AA,
        )

    preview = cv2.resize(
        preview,
        (output_size, output_size),
        interpolation=cv2.INTER_AREA,
    )

    return Image.fromarray(preview)

In [ ]:
@torch.no_grad()
def predict_single_roi_pil(image):
    """
    Prediksi satu ROI PIL image.
    Return:
    - label
    - confidence
    - probabilities
    """
    image = image.convert("RGB")

    image_tensor = efficientnet_preprocess(image).unsqueeze(0).to(DEVICE)

    feature = efficientnet_feature_extractor(image_tensor)
    feature_np = feature.cpu().numpy()

    feature_scaled = effnet_scaler.transform(feature_np)

    feature_tensor = torch.tensor(
        feature_scaled,
        dtype=torch.float32,
    ).to(DEVICE)

    logits = effnet_mlp_model(feature_tensor)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probabilities))
    pred_label = classes[pred_idx]
    pred_confidence = float(probabilities[pred_idx])

    return pred_label, pred_confidence, probabilities


def predict_full_face_3x3_gradio(image):
    """
    Input:
    - foto satu sisi Rubik utuh

    Output:
    - warped face preview
    - prediction grid preview
    - dataframe 9 prediksi
    - summary text
    """
    if image is None:
        empty_df = pd.DataFrame(columns=["position", "row", "col", "predicted_color", "confidence"])
        return None, None, empty_df, "Silakan upload foto satu sisi Rubik terlebih dahulu."

    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    image_rgb = np.array(image.convert("RGB"))

    detection_status = "Auto perspective correction berhasil."

    try:
        quad_points = detect_rubik_face_quad(image_rgb)
        warped_rgb = warp_rubik_face(
            image_rgb,
            quad_points,
            output_size=600,
        )
    except Exception as error:
        warped_rgb = center_square_fallback(
            image_rgb,
            output_size=600,
        )
        detection_status = f"Auto perspective correction gagal. Menggunakan fallback center crop. Detail: {error}"

    rois = split_warped_face_to_rois(
        warped_rgb,
        grid_size=3,
        crop_ratio=0.72,
    )

    predictions = []

    for roi_item in rois:
        label, confidence, probabilities = predict_single_roi_pil(
            roi_item["image"]
        )

        row = roi_item["row"]
        col = roi_item["col"]

        predictions.append({
            "position": f"r{row}c{col}",
            "row": row,
            "col": col,
            "label": label,
            "confidence": confidence,
        })

    result_df = pd.DataFrame([
        {
            "position": item["position"],
            "row": item["row"],
            "col": item["col"],
            "predicted_color": item["label"],
            "confidence": item["confidence"],
        }
        for item in predictions
    ])

    label_grid = np.array([
        [predictions[row * 3 + col]["label"] for col in range(3)]
        for row in range(3)
    ])

    grid_text = "\n".join([
        " | ".join(label_grid[row])
        for row in range(3)
    ])

    warped_image = Image.fromarray(warped_rgb)
    prediction_preview = draw_prediction_grid(
        warped_rgb,
        predictions,
        output_size=600,
    )

    summary = (
        f"{detection_status}\n\n"
        "Predicted 3x3 face:\n"
        f"{grid_text}\n\n"
        "Catatan: hasil terbaik diperoleh jika satu sisi Rubik terlihat penuh, cukup terang, "
        "dan sudut luar bidang Rubik masih terlihat."
    )

    return warped_image, prediction_preview, result_df, summary

In [ ]:
import gradio as gr

with gr.Blocks(title="Speedcube Color Classification") as demo:
    gr.Markdown(
        """
        # Speedcube Color Classification

        Demo klasifikasi warna speedcube menggunakan model **EfficientNetV2-S + MLP**.

        Tersedia dua mode:
        1. Prediksi satu stiker/ROI.
        2. Prediksi satu sisi Rubik utuh dengan perspective correction dan split 3×3.
        """
    )

    with gr.Tab("Single Sticker / ROI"):
        with gr.Row():
            with gr.Column():
                roi_input_image = gr.Image(
                    label="Upload gambar satu stiker/ROI",
                    type="pil",
                )

                roi_predict_button = gr.Button("Prediksi ROI")

            with gr.Column():
                roi_output_label = gr.Label(
                    label="Prediction Probabilities",
                    num_top_classes=6,
                )

                roi_output_table = gr.Dataframe(
                    label="Confidence Detail",
                    headers=["rank", "class", "confidence"],
                )

                roi_output_text = gr.Textbox(
                    label="Summary",
                    lines=4,
                )

        roi_predict_button.click(
            fn=predict_speedcube_color_gradio,
            inputs=roi_input_image,
            outputs=[roi_output_label, roi_output_table, roi_output_text],
        )

    with gr.Tab("Full Face 3x3"):
        with gr.Row():
            with gr.Column():
                full_face_input = gr.Image(
                    label="Upload foto satu sisi Rubik",
                    type="pil",
                )

                full_face_predict_button = gr.Button("Prediksi 3×3")

            with gr.Column():
                warped_output = gr.Image(
                    label="Warped Face",
                    type="pil",
                )

                grid_output = gr.Image(
                    label="Prediction Grid",
                    type="pil",
                )

        full_face_output_table = gr.Dataframe(
            label="Prediksi 9 Blok",
            headers=["position", "row", "col", "predicted_color", "confidence"],
        )

        full_face_output_text = gr.Textbox(
            label="Summary",
            lines=8,
        )

        full_face_predict_button.click(
            fn=predict_full_face_3x3_gradio,
            inputs=full_face_input,
            outputs=[
                warped_output,
                grid_output,
                full_face_output_table,
                full_face_output_text,
            ],
        )

    gr.Markdown(
        """
        ## Catatan Demo

        Mode Full Face 3×3 menggunakan deteksi bidang Rubik, perspective transform,
        lalu membagi hasil warp menjadi 9 ROI. Setiap ROI diprediksi menggunakan
        EfficientNetV2-S pretrained sebagai feature extractor dan MLP head sebagai classifier.
        """
    )

In [ ]:
print("demo" in globals())

if "demo" in globals():
    print(type(demo))
else:
    print("Variabel demo belum ada. Jalankan ulang cell pembuatan Gradio app.")

True
<class 'gradio.blocks.Blocks'>


In [ ]:
demo.launch(
    share=True,
    debug=True,
    inline=True,
    show_error=True,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fce8d11d888fabfdde.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 62, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error